# 📊 Exploratory Data Analysis & Visualization

Notebook 01: Analyze the synthetic spectral dataset and visualize patterns across disease classes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
sys.path.insert(0, '..')

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries loaded ✓')

## 1. Generate & Load Data

In [ ]:
from src.data.data_generator import generate_dataset, BANDS, LABEL_NAMES
from src.features.spectral_indices import compute_indices

# Generate dataset (or load from CSV if already generated)
import os
csv_path = '../data/raw/spectral_dataset.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f'Loaded existing dataset: {csv_path}')
else:
    df = generate_dataset(n_samples=500, seq_len=30)
    os.makedirs('../data/raw', exist_ok=True)
    df.to_csv(csv_path, index=False)
    print(f'Generated and saved: {csv_path}')

# Compute vegetation indices
df = compute_indices(df)
print(f'\nDataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 2. Label Distribution

In [ ]:
labels = df.groupby('sample_id')['label'].first()
label_counts = labels.value_counts().sort_index()
label_names = [LABEL_NAMES[i] for i in label_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[0].bar(label_names, label_counts.values, color=colors)
axes[0].set_title('Disease Class Distribution', fontsize=14)
axes[0].set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(label_counts.values, labels=label_names, colors=colors,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Proportions', fontsize=14)

plt.tight_layout()
plt.show()

## 3. Spectral Band Distributions by Class

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, band in enumerate(BANDS):
    ax = axes[idx // 3, idx % 3]
    for label in [0, 1, 2]:
        data = df[df['label'] == label][band]
        ax.hist(data, bins=50, alpha=0.5, label=LABEL_NAMES[label],
                color=colors[label])
    ax.set_title(band, fontsize=12)
    ax.legend(fontsize=8)
    ax.set_xlabel('Reflectance')

plt.suptitle('Spectral Band Distributions by Disease Class', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 4. Temporal Progression (Healthy vs Diseased)

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=['Healthy', 'Early Stress', 'Active Disease'])

for label in [0, 1, 2]:
    sample = df[(df['label'] == label)].groupby('sample_id').first().index[0]
    sample_data = df[df['sample_id'] == sample].sort_values('timestamp')
    
    for band in ['nir_842', 'red_edge_783', 'swir_1610']:
        fig.add_trace(
            go.Scatter(x=sample_data['timestamp'], y=sample_data[band],
                       name=band, showlegend=(label == 0)),
            row=1, col=label + 1
        )

fig.update_layout(title='Spectral Band Progression Over Time', height=400)
fig.show()

## 5. Vegetation Index Correlation Heatmap

In [ ]:
index_cols = ['NDRE', 'NDVI', 'MSI', 'SWIR_ratio', 'REIP_approx', 'CRI']
all_features = BANDS + index_cols

corr = df[all_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 6. GPS Sample Locations

In [ ]:
sample_locs = df.groupby('sample_id').first().reset_index()

fig = px.scatter_mapbox(
    sample_locs, lat='lat', lon='lon', color='label',
    color_continuous_scale=['green', 'yellow', 'red'],
    title='Sample GPS Locations',
    zoom=10, height=500
)
fig.update_layout(mapbox_style='carto-positron')
fig.show()

## 7. Statistical Tests (ANOVA)

In [ ]:
from scipy import stats

print('One-way ANOVA: Feature differences between disease classes\n')
print(f'{"Feature":<20} {"F-statistic":<15} {"p-value":<15} {"Significant?"}')
print('-' * 65)

for feature in all_features:
    groups = [df[df['label'] == l][feature].values for l in [0, 1, 2]]
    f_stat, p_val = stats.f_oneway(*groups)
    sig = '✓ YES' if p_val < 0.01 else '✗ No'
    print(f'{feature:<20} {f_stat:<15.2f} {p_val:<15.2e} {sig}')